# Clase 18: Compilación, Paralelismo y Computación Distribuida

**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**

**Profesor: Pablo Badilla**

## Objetivos de la clase:

- Aprender a optimizar código a través de `JIT`.
- Comprender el paralelismo de tareas.
- Aprender a paralelizar tareas por medio de funciones en `Joblib`
- Comprender la idea general de computación distribuida.
- Analizar las opciones para computación distribuida: `Dask`.
- Conocer `Polars` como alternativa de alto rendimiento a `Pandas`.
- Comprender la importancia de los tipos de datos y su impacto en memoria.
- Aprender a medir y optimizar el rendimiento mediante *profiling* (memoria y tiempo).
- Conocer herramientas de cómputo acelerado por GPU: CuPy, RAPIDS y frameworks de IA.
- Comprender el uso de asincronía en aplicaciones de ML e IA.

## Motivación

El flujo de trabajo en ciencia de datos consta de **numerosas rutinas de carga, procesamiento y visualización**. Lo ideal es que, una vez pasemos el proceso de experimentación, diseñemos estas operaciones de la forma más óptima posible con el fin de reducir recursos, tiempos de carga utilizados y, obviamente, sus costos asociados.

Pero, ¿por qué debería importarnos la optimización?

---

> **Pregunta ❓:** Si un pipeline de ML tarda 8 horas y lo optimizas a 2 horas, ¿cuánto dinero ahorras al mes en AWS?

**Respuesta:**

- EC2 p3.2xlarge (GPU): ~$3/hora
- 8h → 2h = 6h ahorradas/día
- 6h × $3 × 30 días = **$540/mes por persona**
- Equipo de 5: **$2,700/mes**
- En 2 años: **$64,800 USD**

---

> **Pregunta ❓:** Si puedes probar 2 experimentos por día porque cada uno tarda 4 horas, ¿cuántos experimentos probarás en un sprint de 2 semanas? ¿Y si cada experimento tardara 5 minutos?

**Respuesta:**

- 4 horas: 2 experimentos/día × 10 días = **20 experimentos**
- 5 minutos: 96 experimentos/día × 10 días = **960 experimentos**
- **48x más experimentos** = mejor modelo al final

---

> **Pregunta ❓:** ¿Creen que optimizar código es solo para ingenieros de software, o también es importante para data scientists?

**Respuesta:** Data scientists necesitan optimizar **más**, porque:

- Iteran constantemente (más optimización = más experimentos)
- Trabajan con datos grandes (memoria importa)
- Sus notebooks son la base de producción

---

> **Pregunta ❓:** ¿Saben cuánta energía consume entrenar un modelo de IA grande?

**Respuesta:**

- GPT-3: **552 toneladas de CO₂** (equivalente a 125 autos por un año)
- Entrenar un modelo de visión: **~100 kWh**
- Cada optimización reduce tu impacto ambiental

> *"No optimizar código es como dejar la luz encendida en toda la casa"*


## Lenguajes de Programación

El lenguaje de máquina es el conjunto de instrucciones que el hardware es capaz de interpretar y procesar.
A través de estas instrucciones podemos lograr que nuestro procesador ejecute distintos tipos de acciones muy básicas. 
Este conjunto de lenguajes es comúnmente conocido como *lenguaje de bajo nivel*

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/codigo_maquina.png' width=400 />
</center>

<center>Por suerte no tenemos que siquiera pensar en esto...</center>
    
<center> 
    Fuente: <a href='https://en.wikipedia.org/wiki/Machine_code#/media/File:W65C816S_Machine_Code_Monitor.jpeg'>Wikipedia </a>
</center>
    
    
    


---

## Lenguajes Compilados vs Interpretados

Existen dos enfoques principales para convertir un código de lenguaje de alto nivel a uno de bajo nivel: que el lenguaje sea **Compilado** o **Interpretado**.


<br>
<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/tipos_lenguajes.png' width=800 />
</center>


---

## Computación de alto Rendimiento con Python

Python es utilizado transversalmente, ya sea en la industria o en la academia. Dentro de sus cualidades se encuentra la portabilidad de código, sintaxis intuitiva, disponibilidad de herramientas y documentación. Sin embargo, al ser un lenguaje interpretado se pierden ciertas características intrínsecas de los lenguajes de bajo nivel como C, C++ y Fortran.

En esta y la próxima clase estudiaremos distintas herramientas para mejorar el rendimiento del intérprete, como el uso eficiente de objetos base y la aplicación de técnicas de paralelismo y compilación utilizando tanto librerías nativas, como desarrolladas por terceros. 


> **Pregunta ❓:** ¿Será conveniente programar siempre pensando crear código óptimo?

---

## Optimización del Código

Como directriz general, se recomienda llevar el proceso de desarrollo en dos etapas:

1. La primera consiste en **generar código correcto, comprensible y mantenible**, evitando la sobre-optimización prematura de código.

2. Como segunda etapa, se recomienda comenzar con los procesos de **optimización de código**. Esto se debe a que las herramientas que permiten mejorar los aspectos computacionales interfieren en la sencillez del código, entorpeciendo los procesos de depuración y mantención.

---

> **Pregunta ❓:** Ya tenemos código correcto y queremos optimizarlo. ¿Por dónde empezamos? ¿Cómo sabemos qué parte del código es la que más nos está costando?

**Respuesta:** 

**Necesitamos medir antes de optimizar.**

Sin medición, corremos el riesgo de optimizar partes irrelevantes del código mientras el verdadero cuello de botella queda intacto. A este proceso de medición y localización de zonas críticas se le llama **profiling**.

Veremos dos tipos:

- **Profiling de memoria**: Entender qué tipos de datos usamos y cómo impactan en el consumo de RAM.
- **Profiling de tiempo**: Medir y encontrar los cuellos de botella en la ejecución de nuestro código.

---

### Profiling de Memoria y Tipos de Datos

Los **tipos de datos** tienen un impacto directo en el consumo de memoria y, por ende, en el rendimiento de nuestro código. Elegir el dtype adecuado puede reducir el uso de memoria en un factor de 8x o más.


---

> **Pregunta ❓:** Un CSV de 5GB. ¿Cuánta memoria necesitas para leerlo con pandas?

**Respuesta típica: ~10-15GB**

¿Cómo es posible que un archivo de 5GB ocupe 15GB en RAM?
- pandas usa `int64`/`float64` por defecto (8 bytes por valor)
- Los objetos Python tienen overhead adicional
- Strings se almacenan como `object` (muy ineficiente)

> **Pregunta ❓:** ¿Cuánta memoria consume un DataFrame con 1 millón de filas si cada columna es int64 vs int8?

| dtype | Bytes × 1M filas | Memoria total |
|---|---|---|
| int64 | 8 bytes | **8 MB** |
| int32 | 4 bytes | 4 MB |
| int16 | 2 bytes | 2 MB |
| int8 | 1 byte | **1 MB** |

**8x menos memoria** solo cambiando el dtype. Con 100 columnas: de 800MB a 100MB.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import gc

n = 10_000_000  # 10M filas para que las diferencias sean visibles

df = pd.DataFrame(
    {
        "int64": np.random.randint(0, 100, n, dtype=np.int64),
        "int32": np.random.randint(0, 100, n, dtype=np.int32),
        "int16": np.random.randint(0, 100, n, dtype=np.int16),
        "int8": np.random.randint(0, 100, n, dtype=np.int8),
        "float64": np.random.rand(n).astype(np.float64),
        "float32": np.random.rand(n).astype(np.float32),
        "category": pd.Categorical([f"grupo_{i % 20}" for i in range(n)]),
        "object": [f"grupo_{i % 20}" for i in range(n)],
    }
)

# Memoria por columna en MB (deep=True para contar strings reales)
mem = df.memory_usage(deep=True).drop("Index") / 1e6

print(f"{'Columna':>10}  {'MB':>8}  {'bytes/fila':>12}")
print("-" * 36)
for col, mb in mem.items():
    bpf = mb * 1e6 / n
    print(f"{col:>10}  {mb:>8.2f}  {bpf:>12.1f}")
print("-" * 36)
print(f"{'TOTAL':>10}  {mem.sum():>8.2f}")

# Gráfico de barras con Plotly
fig = go.Figure(
    go.Bar(
        x=mem.index.tolist(),
        y=mem.values,
        text=[f"{v:.1f} MB" for v in mem.values],
        textposition="outside",
        marker_color=[
            "#ef553b"
            if "64" in c
            else "#ffa15a"
            if "32" in c
            else "#19d3f3"
            if c in ("int16", "float16")
            else "#00cc96"
            if c == "int8"
            else "#636efa"
            if c == "category"
            else "#ab63fa"
            for c in mem.index
        ],
    )
)
fig.update_layout(
    title="Memoria por columna — 10M filas",
    xaxis_title="dtype",
    yaxis_title="MB",
    yaxis=dict(rangemode="tozero"),
    template="plotly_white",
    height=420,
)
fig.show()

# Liberar recursos
del df, mem, fig
gc.collect();


#### Consejos prácticos para pandas

```python
# Usar category para columnas con pocos valores únicos
df['genero'] = df['genero'].astype('category')  # 2 valores -> ~95% menos memoria

# Usar float32 en lugar de float64 para ML
X = X.astype(np.float32)  # 50% menos memoria, precisión suficiente para ML

# Downcast enteros automáticamente
df['edad'] = pd.to_numeric(df['edad'], downcast='integer')

# Leer con dtypes específicos para ahorrar memoria
df = pd.read_csv('datos.csv', dtype={'id': 'int32', 'precio': 'float32'})
```


| dtype | Bytes | Rango aproximado | Cuándo usarlo |
|---|---|---|---|
| `int8` | 1 | -128 a 127 | Edades, calificaciones |
| `int16` | 2 | -32K a 32K | Contadores pequeños |
| `int32` | 4 | -2B a 2B | IDs, enteros generales |
| `int64` | 8 | ±9 quintillones | Por defecto (innecesario la mayoría) |
| `float32` | 4 | ~7 dígitos | ML, datos científicos |
| `float64` | 8 | ~15 dígitos | Precisión金融 (default pandas) |
| `category` | variable | — | Strings con pocos valores únicos |

---

#### Conexión con el mundo real: cuantización en LLMs

Los tipos de datos no son solo un detalle técnico de pandas — son el mecanismo central detrás de una de las técnicas más importantes para hacer LLMs accesibles: la **cuantización**.

Un modelo como LLaMA 3 8B tiene ~8.000 millones de parámetros. Si cada uno se almacena en `float32` (4 bytes), el modelo ocupa **32 GB** de VRAM. Con cuantización se reduce la precisión de esos pesos:

| Formato | Bytes/param | LLaMA 3 8B | Calidad |
|---------|------------|------------|--------|
| `float32` | 4 | ~32 GB | Referencia |
| `float16` / `bfloat16` | 2 | ~16 GB | Casi idéntica |
| `int8` (Q8) | 1 | ~8 GB | Muy buena |
| `int4` (Q4) | 0.5 | ~4 GB | Buena (algo de pérdida) |
| `int2` (Q2) | 0.25 | ~2 GB | Degradación notable |

**¿Por qué funciona?** Las redes neuronales son sorprendentemente robustas a la reducción de precisión en los pesos. El modelo no necesita saber que $w = 0.31415926...$; con $w \approx 0.31$ el resultado es prácticamente el mismo.

> **Trade-off:** Menos bits → menos VRAM y más velocidad de inferencia, pero mayor pérdida de precisión numérica. Q4 es hoy el estándar de facto para correr LLMs en hardware de consumo (llama.cpp, Ollama, HuggingFace GGUF).

### Medición del Tiempo de Ejecución ⏰


El tiempo de ejecución es el tiempo tomado por algún segmento de código, función en completar su ejecución.

En Python, la forma más sencilla de medir el tiempo de ejecución es a través de la librería `time`. El ejemplo siguiente muestra como utilizarla.

In [ ]:
import time
from math import cos, sin

Definimos un rango de datos a operar 

In [ ]:
x = [0.1 * i for i in range(100000)]

x[0:10]  # veamos los datos

Luego definimos la función que mediremos. Esta simplemente calcula $(\sin(val) + \cos(val)^2)^{1/3}$ y luego retorna su valor.

In [ ]:
def func_1(val):
    return (sin(val) + cos(val) ** 2) ** (1 / 3)

Ahora, estudiamos el tiempo de ejecución por medio de la función `process_time`.

In [ ]:
# tiempo inicial
t0 = time.process_time()

for i in x:
    func_1(i)

# tiempo final
t1 = time.process_time()

# el tiempo transcurrido es simplemente el delta entre t1 y t0
print("Tiempo transcurrido", t1 - t0)

> **Pregunta ❓:** ¿Si ejecutamos nuevamente la celda anterior, obtendremos los mismos tiempos? ¿Existirá alguna forma más consistente de medir el tiempo de ejecución del código?

---

### `timeit`

En algunas ocasiones se desea medir el tiempo de ejecución para tareas sencillas, la librería estándar de Python provee el módulo `timeit`. En la práctica, una llamada de `timeit` ejecuta por defecto 10.000.000 veces el código (variable según cuánto se demore el proceso) y repite 7 veces el experimento. Luego reporta el tiempo de ejecución promedio.

Este puede ser utilizado directamente en la consola interactiva IPython o en notebooks de Jupyter por medio del comando mágico `%timeit` para el caso de una línea de código y `%%timeit` para medir toda la celda. 

Documentación de `%timeit`: [Timeit Magic en la documentación de Ipython](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit)



**Ejemplo**


Medimos la eficiencia de la implementación original de python de `cos`.

In [ ]:
%timeit cos(0.5)

Y lo comparamos con el tiempo de ejecución promedio para la función coseno de `numpy`.

In [ ]:
import numpy as np

In [ ]:
%timeit np.cos(0.5)

In [ ]:
%timeit func_1(100)

---

> **Pregunta ❓:** Si tu código tarda 10 minutos, ¿dónde crees que está el problema?

La mayoría dice: "en el loop más grande" o "en el cálculo más complejo".

**Sorpresa:** El 80% del tiempo suele estar en operaciones que parecen simples, como:
- Acceso a diccionarios
- Llamadas a funciones pequeñas
- Operaciones de I/O (lectura de archivos)

---

## Calcular el valor de $\pi$ usando Montecarlo


Idea: 

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/montecarlo.png' width=300 />
<div/>
    
$$\frac{\text{area círculo}}{\text{area cuadrado}} = \frac{\pi r^2}{(2r)^2} $$

$$ 4* \frac{\text{area círculo}}{\text{area cuadrado}} = \pi $$


Y después simulamos que lanzamos puntos al azar a nuestra figura y contamos: 

$$ 4* \frac{\text{puntos en el circulo}}{\text{puntos en el cuadrado}} = \pi $$



> **Nota pedagógica:** A partir de aquí, **el problema de Monte Carlo es nuestro hilo conductor**. Lo usaremos como benchmark estándar para comparar cada técnica de optimización que veremos:
>
> | Paso | Técnica | Objetivo |
> |------|---------|----------|
> | 1 | **Profiling** | Identificar dónde está el cuello de botella |
> | 2 | **NumPy** | Reemplazar loops Python por operaciones vectorizadas |
> | 3 | **Numba JIT** | Compilar el loop crítico a código nativo |
> | 4 | **Paralelismo** | Distribuir el trabajo en múltiples cores |
>
> Esto nos permitirá **medir el speedup acumulado** a medida que avanzamos.


---

### Profiling: encontrar cuellos de botella

Antes de optimizar, necesitamos saber **dónde** está el problema. El **profiling** nos permite analizar qué partes de nuestro código consumen más tiempo y memoria.

Python incluye `cProfile`, un profiler integrado que muestra cuánto tiempo se ejecuta cada función:

In [ ]:
import cProfile
import random


def monte_carlo_pi_python(nsamples):
    acc = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / nsamples


# Profilear la función
cProfile.run("monte_carlo_pi_python(500_000)", sort="cumulative")


Las columnas clave son:
- **ncalls**: número de llamadas a la función
- **tottime**: tiempo total gastado **dentro** de la función (sin subllamadas)
- **cumtime**: tiempo acumulado (incluyendo subllamadas)
- **filename:lineno**: ubicación del código

En este caso, vemos que `random.random()` es el cuello de botella — se llama 1,000,000 veces y consume la mayor parte del tiempo.

> **Nota:** Para un análisis más visual, puedes usar `snakeviz` (`pip install snakeviz`) que genera un gráfico interactivo del profiling: `snakeviz profile_output`


#### Análisis línea por línea con `line_profiler`

Para ver qué **línea exacta** es la lenta, usamos `line_profiler`:

In [ ]:
# Instalar si no está disponible: pip install line_profiler
try:
    from line_profiler import LineProfiler

    def monte_carlo_pi_python(nsamples):
        acc = 0
        for i in range(nsamples):
            x = random.random()
            y = random.random()
            if (x**2 + y**2) < 1.0:
                acc += 1
        return 4.0 * acc / nsamples

    lp = LineProfiler()
    lp.add_function(monte_carlo_pi_python)
    lp_wrapper = lp(monte_carlo_pi_python)
    lp_wrapper(500_000)
    lp.print_stats()
except ImportError:
    print("line_profiler no instalado. Instalar con: pip install line_profiler")


**Resumen de herramientas de profiling:**

| Herramienta | Qué muestra | Cuándo usarla |
|---|---|---|
| `cProfile` | Tiempo por función | Primera exploración |
| `line_profiler` | Tiempo por línea | Encontrar línea exacta |
| `memory_profiler` | Uso de memoria | Problemas de memoria |
| `snakeviz` | Visualización interactiva | Presentar resultados |

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_COLORES = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]


def plot_comparacion_tiempos(
    resultados, titulo="Comparación de rendimiento", nsamples=None
):
    """
    resultados : dict {nombre: TimeitResult | float}
      - TimeitResult (%timeit -o): barras = promedio ± stdev, diamante = mejor tiempo.
      - float (time.time()): barra simple, sin barras de error.
    titulo     : str  título del gráfico.
    nsamples   : int  opcional, se agrega al título si se indica.
    """
    nombres = list(resultados.keys())
    values = list(resultados.values())

    promedios, stdevs, bests = [], [], []
    for v in values:
        if hasattr(v, "average"):  # TimeitResult
            promedios.append(v.average)
            stdevs.append(v.stdev)
            bests.append(v.best)
        else:  # tiempo único (float)
            t = float(v)
            promedios.append(t)
            stdevs.append(None)
            bests.append(t)

    baseline = bests[0]
    speedups = [baseline / b for b in bests]
    colores = _COLORES[: len(nombres)]
    es_agr = any(s is not None for s in stdevs)

    titulo_completo = titulo
    if nsamples:
        titulo_completo += f" — {nsamples:,} muestras"

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Tiempo de ejecución", f"Speedup vs {nombres[0]}"),
    )

    # ── barras de tiempo ──
    fig.add_trace(
        go.Bar(
            x=nombres,
            y=promedios,
            marker_color=colores,
            error_y=(
                dict(
                    type="data",
                    array=[s if s is not None else 0 for s in stdevs],
                    visible=True,
                    color="#555",
                    thickness=1.5,
                    width=6,
                )
                if es_agr
                else None
            ),
            text=[
                f"best: {b * 1e3:.2f} ms<br>avg: {a * 1e3:.2f} ms<br>σ: {s * 1e3:.2f} ms"
                if s is not None
                else f"{a * 1e3:.2f} ms"
                for b, a, s in zip(bests, promedios, stdevs)
            ],
            hovertemplate="%{x}<br>%{text}<extra></extra>",
            textposition="outside" if not es_agr else "none",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # diamante = mejor tiempo (solo resultados agregados)
    if es_agr:
        fig.add_trace(
            go.Scatter(
                x=nombres,
                y=bests,
                mode="markers+text",
                marker=dict(
                    symbol="diamond",
                    size=10,
                    color="white",
                    line=dict(color="#333", width=1.5),
                ),
                text=[f"{b * 1e3:.2f} ms" for b in bests],
                textposition="top center",
                name="mejor tiempo",
                showlegend=True,
            ),
            row=1,
            col=1,
        )

    # ── speedup ──
    fig.add_trace(
        go.Bar(
            x=nombres,
            y=speedups,
            marker_color=colores,
            text=[f"{s:.1f}×" for s in speedups],
            textposition="outside",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        title_text=titulo_completo,
        height=480,
        margin=dict(t=80),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.15),
    )
    fig.update_yaxes(title_text="Tiempo (s)", row=1, col=1)
    fig.update_yaxes(title_text="Speedup (×)", row=1, col=2)
    fig.show()



---

### Vectorización con NumPy

La **vectorización** es reemplazar loops de Python con operaciones sobre arrays completos. NumPy ejecuta estas operaciones en C, eliminando el overhead del intérprete de Python.

Veámoslo con nuestro ejemplo de Monte Carlo:

In [ ]:
import numpy as np


def monte_carlo_pi_numpy(nsamples):
    x = np.random.rand(nsamples)
    y = np.random.rand(nsamples)
    return 4.0 * np.sum(x**2 + y**2 < 1.0) / nsamples


In [ ]:
t_python = %timeit -o -q monte_carlo_pi_python(100_000)
t_numpy = %timeit -o -q monte_carlo_pi_numpy(100_000)

plot_comparacion_tiempos(
    {"Python (puro)": t_python, "NumPy": t_numpy},
    titulo="Monte Carlo π: Python puro vs NumPy",
    nsamples=100_000,
)



**¿Por qué es más rápido?**

1. **Sin loop de Python**: NumPy ejecuta la operación completa en C, sin iterar en Python
2. **Operaciones por lote**: `x**2 + y**2 < 1.0` se ejecuta para todo el array de una vez
3. **Memoria contigua**: Los arrays NumPy están en memoria contigua, mejorando el cache

> **Regla general:** Si haces un loop sobre datos numéricos, probablemente puedas vectorizarlo con NumPy.


---

Ahora que sabemos **medir** el tiempo de ejecución, la siguiente pregunta es: ¿cómo podemos hacer que nuestro código se ejecute más rápido? Una estrategia poderosa es usar **compilación JIT** (*Just-In-Time*).

---

## Compilación JIT con Numba

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/numba.png' width=600/>
</center>

Un proyecto interesante **es** la librería **`Numba`** la cual está enfocada en **analizar y compilar funciones de Python**. Compiladores como Numba, diseñados para compilar código en ejecución (y no previo a la ejecución) se denominan compiladores **JIT** (just in time). 

Numba permite compilar funciones individuales de Python usando una *máquina virtual de bajo nivel* o LLVM por sus siglas en inglés (LLVM es un conjunto de herramientas pensadas para escribir compiladores).

Por medio de LLVM Numba inspecciona funciones de Python y las compila utilizando una capa de representación intermedia similar a código *assembly*. La potencia de esta inspección radica en la inferencia de tipos de datos generando versiones compiladas con tipos de datos estáticos.

Numba se basa principalmente en el decorador `@jit` con el cual se definen las funciones a compilar.





### Comparativa de implementaciones: Python → NumPy → Numba

Python puro y NumPy ya fueron definidas y medidas en la sección de vectorización. A continuación agregamos las versiones compiladas con **Numba** para completar la comparativa:


### $\pi$ con Montecarlo en `Numba`

Y ahora probamos con una función compilada usando el decorador `@jit`.

In [ ]:
import random

from numba import jit


@jit(nopython=True)
def monte_carlo_pi_numba(nsamples):
    acc = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()

        if (x**2 + y**2) < 1.0:
            acc += 1

    return 4.0 * acc / nsamples

### Costo de compilación JIT
#### Costo de compilación JIT

Numba es un compilador **JIT** (*Just-In-Time*), lo que significa que compila las funciones la **primera vez** que se ejecutan. Esto implica un costo de compilación en la primera llamada que no se repite en llamadas subsecuentes.

Veamos esta diferencia:

In [ ]:
import time

start = time.time()
monte_carlo_pi_numba(100_000)
cold_time = time.time() - start

start = time.time()
monte_carlo_pi_numba(100_000)
warm_time = time.time() - start

print(
    f"Cold start: {cold_time:.4f}s | Warm: {warm_time:.6f}s | Overhead: {cold_time / warm_time:.0f}×"
)

plot_comparacion_tiempos(
    {"Cold start (1ª llamada)": cold_time, "Warm (2ª llamada)": warm_time},
    titulo="Costo de compilación JIT — Numba",
)


### Numba y Numpy

`Numba` también está diseñado para funcionar en conjunto con `numpy`

In [ ]:
@jit(nopython=True)
def monte_carlo_pi_numpy_numba(nsamples):
    acc = 0
    x = np.random.rand(nsamples)
    y = np.random.rand(nsamples)

    op = x**2 + y**2
    dentro_circulo = op[op < 1.0]

    return 4.0 * np.count_nonzero(dentro_circulo) / nsamples

In [ ]:
monte_carlo_pi_numpy_numba(100000)

### Comparación de rendimiento: Monte Carlo π
#### Comparación de rendimiento: Monte Carlo π

Comparamos los tiempos de ejecución de las 4 implementaciones:

In [ ]:
t_numba = %timeit -o -q monte_carlo_pi_numba(100_000)
t_numba_np = %timeit -o -q monte_carlo_pi_numpy_numba(100_000)

plot_comparacion_tiempos(
    {
        "Python (puro)": t_python,
        "NumPy": t_numpy,
        "Python + Numba": t_numba,
        "NumPy + Numba": t_numba_np,
    },
    titulo="Monte Carlo π: comparativa completa",
    nsamples=100_000,
)


### Importante: `Numba` solo compila código de Python y `Numpy`

Está en general diseñado para optimizar tareas matemáticas y con ciclos.
No entiende librerías más complejas como `pandas` por ejemplo.





In [ ]:
import pandas as pd

x = {"a": [1, 2, 3], "b": [20, 30, 40]}

In [ ]:
def use_pandas(a):  # Function will not benefit from Numba jit
    df = pd.DataFrame.from_dict(a)  # Numba doesn't know about pd.DataFrame
    df += 1  # Numba doesn't understand what this is
    return df.cov()  # or this!

In [ ]:
%timeit use_pandas(x)

In [ ]:
@jit(nopython=True)
def use_pandas(a):  # Function will not benefit from Numba jit
    df = pd.DataFrame.from_dict(a)  # Numba doesn't know about pd.DataFrame
    df += 1  # Numba doesn't understand what this is
    return df.cov()  # or this!

In [ ]:
use_pandas(x)

---

La compilación JIT acelera el código en **un solo núcleo**. Pero, ¿qué pasa cuando necesitamos más potencia?

La siguiente frontera es el **paralelismo**: utilizar múltiples núcleos de procesamiento simultáneamente.

| Técnica | ¿Qué optimiza? | Límite |
|---------|---------------|--------|
| NumPy | Evita overhead del intérprete Python | Un solo core |
| Numba JIT | Compila a código nativo | Un solo core |
| **Paralelismo** | **Usa múltiples cores simultáneamente** | Número de cores físicos |
| Distribuido | Usa múltiples máquinas | Presupuesto/infraestructura |

> **Regla:** Primero optimiza el código serial (NumPy/Numba), luego paraleliza. Paralelizar código lento solo da código lento en paralelo.

---

> **Pregunta ❓:** Un dataset tiene 10 millones de filas. Tu laptop tiene 8GB de RAM. ¿Qué haces?

**Opciones:**

1. ❌ Subir la RAM (no siempre es posible)
2. ❌ Comprar una laptop nueva (caro)
3. ✅ Usar herramientas que procesen datos fuera de memoria
4. ✅ Paralelizar para usar todos los cores disponibles
5. ✅ Muestrear para prototipar, luego escalar

> **Pregunta ❓:** Si entrenas un modelo XGBoost con 100GB de datos, ¿cuántos cores necesitas para que termine en 1 hora en vez de 10?

**Respuesta:** Idealmente 10 cores (lineal). En la práctica: 6-8 cores (overhead de coordinación). Por eso se usa `n_jobs=-1`.

---

## Paralelismo

El paralelismo se basa en el uso de múltiples unidades de cómputo de manera simultánea, con el fin de mejorar la eficiencia en rutinas de código. La idea principal consiste en enfrentar un problema de programación, dividiéndolo en subunidades independientes y utilizar los núcleos disponibles de la máquina para resolver tales subunidades en paralelo.

<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/paralelo_vs_secuencial.jpeg'/>
<center>
Fuente: 
<a href='https://towardsdatascience.com/an-intro-to-parallel-computing-with-ray-d8503629485'>https://towardsdatascience.com/an-intro-to-parallel-computing-with-ray-d8503629485</a>
    
</center>



---

### Problemas Data Parallel

Los problemas **Data Parallel** son aquellas en las que se le aplica una función particular sobre todos los datos (por ejemplo, multiplicar una matriz por un escalar).


En este tipo de problema paralelizable, es importante que la función sea exactamente la misma y que el calculo de esta es independiente de todas las otras funciones. Por lo mismo, estas tareas también son denominadas **perfectamente paralelizables**. 

Las operaciones elemento por elemento sobre arreglos poseen esta propiedad. 


<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/cpu_gpu.jpg' width=500 />
</div>

<div align='center'>
    Fuente: 
<a href='https://www.nvidia.com/es-la/drivers/what-is-gpu-computing/'>Nvidia.</a>
</div>

  
Imaginense la cantidad de operaciones simples que una GPU puede lograr hacer en paralelo. Por ejemplo, sumar una matriz con otra elemento a elemento.
  



---

### Problemas Task Parallel

Los problemas task parallel son aquellos que ejecutan varias tareas distintas en distintos hilos/procesos sobre distintos procesadores.

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/paralelismo_memoria.png' width=500 />
</center>

<center>
Fuente:    
<a href='https://manningbooks.medium.com/explaining-mapreduce-with-ducks-f643c78e0b40'>https://manningbooks.medium.com/explaining-mapreduce-with-ducks-f643c78e0b40</a>
</center>

<br>


Por lo general, este tipo de tasks no son completamente independientes y necesitan compartir información. En estos casos, se debe tener en cuenta que la comunicación entre subunidades y los datos compartidos **quitan eficiencia** al problema que se resuelve, pues se incurre en *costos de comunicación*. 

La comunicación entre procesos es inherentemente costosa y puede llevar a fallas de correctitud. Por lo general, se enfrenta el problema de costo de comunicación y correctitud del manejo de memoria por medio de sistemas que se comunican por medio de **threads/hilos con memoria compartida** y **procesos con memoria distribuida**.

---

### Hilos de Procesamiento o Threads

En el caso de memoria compartida, las subunidades involucradas en el programa tienen acceso a un espacio común de memoria, este por lo general es de acceso rápido. 

Si bien esto solventa el problema de velocidad de comunicación, el problema de correctitud sigue latente, por lo que se hace necesario utilizar técnicas de **sincronización**. 

La manera usual en la que se implementan procesos de memoria compartida es por medio de **threads o hilos de ejecución**. Estos consisten en subtareas originadas de un proceso en particular y que comparten recursos. 


<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/threads.jpg' width=500/>
</center>

<center>
Fuente:
<a href='https://www.cs.uic.edu/~jbell/CourseNotes/OperatingSystems/4_Threads.html'> https://www.cs.uic.edu/~jbell/CourseNotes/OperatingSystems/4_Threads.html </a>
</center>

---

### Procesos

Por otra parte, el concepto de memoria distribuida concibe cada subunidad como un proceso completamente separado del resto con su propio espacio de memoria asociado. En este caso, la comunicación entre procesos se debe manejar de manera explicita y es más costosa que en el caso de memoria compartida, sin embargo, se reduce el riesgo de generar errores en el manejo de memoria. 

Este tipo de paralelismos puede ser observadas en los distintos procesos que ejecuta nuestro computador.

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/thread_process.png' width=500/>
</center>


<center>
    Fuente:
    <a href='https://www.javamex.com/tutorials/threads/how_threads_work.shtml'>https://www.javamex.com/tutorials/threads/how_threads_work.shtml</a>
</center>

    
<br>
<br>

> **Pregunta ❓**: ¿Qué aplicación de data science podría beneficiarse de la aplicación de procesos paralelos?

---

### Threads y Procesos en Python

Python puede manejar threads pero dado el diseño de su intérprete, por defecto, se puede ejecutar solo una tarea a la vez, esto se conoce como **GIL** (Global Interpreter Lock). GIL provoca que cada vez que un hilo ejecute una orden de Python, se genere un bloqueo que solo será liberado una vez la ejecución del hilo termine.

> **Esto hace que los hilos solo puedan ser ejecutados de manera secuencial.**

Es decir, Python no puede ejecutar 2 o más hilos de ejecución al mismo tiempo usando más de un procesador.


Aunque GIL evita la ejecución de hilos usando múltiples procesadores en paralelo, es posible utilizar procesos mediante algunas librerías. La principal es `multiprocessing`

Multiprocessing ofrece una interfaz sencilla que incluye múltiples herramientas para manejar sincronización y ejecución de tareas. Es posible importar esta librería de manera estándar. 

```python
import multiprocessing
```

Es posible crear procesos independientes por medio de la clase `Process`, para ello basta extender el método `__init__` para inicializar los datos a procesar y generar el método `run` sobre el cual se ejecuta el proceso.

**Ejemplo**
 
Se genera un proceso independiente utilizando la clase `Process`

**Nota (Python ≥ 3.13/3.14):**
- **Free-threaded (3.13+):** existe el [modo sin GIL](https://docs.python.org/3.13/whatsnew/3.13.html#free-threaded-cpython) (PEP 703), aún experimental.
- **Start method (3.14):** el default cambió de `fork` a `forkserver` en Linux. Con `forkserver`, los procesos hijos re-importan el módulo principal — lo que falla en Jupyter porque las clases definidas en celdas no son importables. La celda siguiente fuerza `fork`, que copia la memoria del padre directamente.

In [ ]:
import sys
import multiprocessing

if sys.version_info >= (3, 14):
    # Python 3.14+ cambia el default a 'forkserver': los procesos hijos
    # re-importan el módulo principal, lo que falla en Jupyter.
    multiprocessing.set_start_method("fork", force=True)
    print(
        f"Python {sys.version_info.major}.{sys.version_info.minor}: forzando start method → 'fork'"
    )
else:
    # ≤ 3.13: 'fork' ya es el default en Linux, no se requiere cambio.
    print(
        f"Python {sys.version_info.major}.{sys.version_info.minor}: start method por defecto → '{multiprocessing.get_start_method()}'"
    )


In [ ]:
import time
from multiprocessing import Process


class Proceso_independiente(Process):
    def __init__(self, num):
        super().__init__()
        self.num = num

    def run(self):
        print("Mi número:", self.num, "\nMe voy a dormir 10s 💤😴💤")
        time.sleep(10)
        print("Desperté 😃")

Para utilizar el proceso se instancia un objeto de la clase `Proceso_independiente` y se llama el método `.start()` 

In [ ]:
proc = Proceso_independiente(5)
proc.start()

In [ ]:
proc = Proceso_independiente(10)
proc.start()

In [ ]:
print("¿¿¿🤨??? Me puedo ejecutar sin esperar a que la celda anterior termine")

En el caso en que se requiera esperar la finalización de un conjunto de tareas paralelas para luego recopilar resultados, es posible utilizar el método `.join()`.

In [ ]:
proc = Proceso_independiente(5)
proc.start()
proc.join()

print("Aquí tuve que esperar 😔")

Con la construcción actual, es posible levantar tantos procesos como se requiera, en esta caso se levantan 3 procesos.

> **Nota:** A continuación redefinimos `Proceso_independiente` con mensajes más claros para facilitar la observación del comportamiento paralelo. La lógica interna (sleep + print) es la misma.

In [ ]:
import time
from multiprocessing import Process


class Proceso_independiente(Process):
    def __init__(self, num):
        super().__init__()
        self.num = num

    def run(self):
        print(f"Me voy a dormir 10s ({self.num})💤😴💤\n")
        time.sleep(10)
        print(f"Desperté ({self.num})😃\n")

In [ ]:
# Se definen los 3 procesos
procesos = (
    Proceso_independiente(1),
    Proceso_independiente(2),
    Proceso_independiente(3),
)

# Se mide el tiempo de ejecucion
start = time.time()

# Iniciar todos los procesos
for p in procesos:
    p.start()

# Esperar a que terminen todos los procesos
for p in procesos:
    p.join()

end = time.time()


print("Tiempo de ejecución: ", end - start)

Estos tres procesos corren de manera paralela, pues su tiempo de ejecución total es aproximado al tiempo de ejecución individual. 

Es necesario comprender que el orden de ejecución de procesos paralelos no es necesariamente ordenado y predecible pues depende de cómo el sistema operativo asigne los recursos. 

### Memoria Compartida y Dataraces

Un data race es una situación que ocurre cuando uno o más hilos acceden concurrentemente a una posición de memoria o variable, al menos uno está escribiendo y al menos uno no está sincronizado con los otros hilos.

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/datarace_1.png' width=400/>
</div>

<div align='center'>
    Ejecución secuencial en memoria compartida por threads.
    Fuente: <a href='https://en.wikipedia.org/wiki/Race_condition'>Wikipedia</a>
</div>

<br>
<br>

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/datarace_2.png' width=400/>
</div>

<div align='center'>
    Ejecución paralela en memoria compartida por threads.
    Fuente: <a href='https://en.wikipedia.org/wiki/Race_condition'>Wikipedia</a>
</div>

<br>

**La solución es tener mecanismos de sincronización** de hilos. 



### Ejemplo en `multiprocessing`


El comportamiento predeterminado de `multiprocessing` es generar procesos con memoria independiente, sin embargo, permite definir ciertas variables en memoria compartida. Para definir una variable en memoria compartida se utiliza la clase `Value`, a esta clase se le entrega un tipo de dato que puede ser `i` para entero, `f` para flotante, `d` para doble precisión entre otros. 


In [ ]:
from multiprocessing import Value

comp_var = Value("d")

Al utilizar variables en memoria compartida se deben tener en cuenta los procesos que acceden a ella, manejando la *concurrencia*, es decir, si los procesos pueden acceder a dichas variables de manera simultánea u ordenada. Por lo general en la actualización de valores unidimensionales se debe tener en cuenta la concurrencia bloqueando el acceso simultaneo. En arreglos se puede permitir tal manipulación siempre que los cómputos sean independientes. 

Para bloquear el acceso a una variable compartida se hace uso de la clase `Lock`.

In [ ]:
from multiprocessing import Lock

lock = Lock()

A continuación se genera una rutina que accede a una variable de memoria compartida

In [ ]:
from multiprocessing import Process, Value


class Process_shared(Process):
    def __init__(self, var, n=10000):
        super().__init__()
        self.var = var
        self.n = n

    def run(self):
        for i in range(self.n):
            self.var.value += 1

El proceso asociado toma un valor y le añade 1 hasta `n = 10000` veces por proceso. Se crea el valor inicial y se inicializan 3 procesos

In [ ]:
def test():
    var = Value("i")
    var.value = 0

    procs = [Process_shared(var) for i in range(3)]

    for p in procs:
        p.start()

    for p in procs:
        p.join()

    print(var.value)

Se prueba el resultado

In [ ]:
test()

Como se puede ver, el resultado no es necesariamente 30.000, esto se debe al acceso simultaneo y aleatorio de los procesos a `var`, para solucionar este problema se hace uso de `lock`, para ello se redefine la clase `Process_shared` observando que lock es un *context manager*

In [ ]:
class Process_shared_lock(Process):
    def __init__(self, var, n=10000):
        super().__init__()
        self.var = var
        self.n = n

    def run(self):
        for i in range(self.n):
            with lock:
                self.var.value += 1

Se redefine la prueba asociada y se ejecuta:

In [ ]:
def test():
    var = Value("i")
    var.value = 0

    procs = [Process_shared_lock(var) for i in range(3)]

    for p in procs:
        p.start()
    for p in procs:
        p.join()

    print(var.value)


test()

Con lo cual se obtiene el resultado buscado.

Sin embargo, coordinar procesos más complejos se torna tedioso y complejo, además de ser susceptible a errores.
Por lo general, se recomienda, a menos que sea estrictamente necesario, a librerías que facilitan la paralelización, como las que vamos a ver a continuación.

---

### `multiprocessing.Pool`: paralelismo funcional

La librería estándar de Python ofrece `multiprocessing.Pool`, que permite distribuir trabajo entre múltiples procesos usando `map`, `starmap` y otros métodos. Si bien es una herramienta válida, para ciencia de datos se recomienda **Joblib** (que veremos a continuación), ya que ofrece una API más sencilla, persistencia de resultados y está optimizado para tareas de ML.

> **Nota:** Si necesitas control fino del paralelismo, `multiprocessing.Pool` es una opción. Pero para la mayoría de los casos en ciencia de datos, Joblib es la elección natural.


---

### Paralelización con `Joblib`



<div align='center'>
    <img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/joblib.png' width=200>
</div>

Otra forma de paralelizar de forma relativamente sencilla es usar la librería `joblib`. 
Esta permite ejecutar funciones de forma paralela, pero ahora de manera funcional. 
Es decir, le entregamos una función y una lista de argumentos y ejecuta una función con dichos argumentos de forma paralela.

Para esto, utiliza el decorador `delayed` sobre una función (lo que la transforma a lazy, es decir, no se ejecuta instantáneamente). Luego a través del objeto `Parallel` que toma el número de trabajos concurrentes que se ejecutarán (`n_jobs`) ejecuta las funciones con sus parámetros.

El siguiente ejemplo veremos como paralelizar el cálculo de coseno sobre una lista:

In [ ]:
import numpy as np

[np.cos(i) for i in np.arange(0, 100, 0.1)]

La notación es muy similar a un list comprehension, solo que con 2 diferencias:

- Se reemplazan los corchetes exteriores `[f(i) for i in ...]` por paréntesis `(f(i) for i in ...)` (Esto da lugar a un generador en vez de una lista).
- Se encapsula la función a aplicar a cada elemento con la función `delayed`, o sea, `f(i)` por `delayed(f)(i)`.

Luego, lo anterior se le pasa como argumento a un Parallel.


In [ ]:
from math import sqrt
import numpy as np

from joblib import Parallel, delayed

# n_jobs=-1 indica que se usarán todos los procesadores disponibles.
Parallel(n_jobs=-1)(delayed(np.cos)(i) for i in np.arange(0, 100, 0.1))

In [ ]:
t_cos_seq = %timeit -o [np.cos(i) for i in np.arange(0, 1, 0.001)]


In [ ]:
t_cos_par = %timeit -o Parallel(n_jobs=-1)(delayed(np.cos)(i) for i in np.arange(0, 1, 0.001))


In [ ]:
plot_comparacion_tiempos(
    {"Secuencial": t_cos_seq, "Joblib (n_jobs=-1)": t_cos_par},
    titulo="CPU bound ligero (np.cos): overhead de paralelización con Joblib",
)


Para tareas numéricas no es tan efectivo (ya existe un cierto overhead/coste de generar los subprocesos), pero para tareas pesadas, se comporta bastante bien.

Para este ejemplo, leeremos 10 veces un archivo con números aleatorios en forma secuencial y en forma paralelizada:

In [ ]:
import pandas as pd


def leer_archivo(_):
    _ = pd.read_csv(
        "https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo//num_aleatorios.csv"
    )

In [ ]:
t_leer_seq = %timeit -o [leer_archivo(_) for _ in range(0, 10)]


In [ ]:
t_leer_par = %timeit -o Parallel(n_jobs=-1)(delayed(leer_archivo)(_) for _ in range(0, 10))


In [ ]:
plot_comparacion_tiempos(
    {"Secuencial": t_leer_seq, "Joblib (n_jobs=-1)": t_leer_par},
    titulo="I/O bound: lectura de archivos — secuencial vs Joblib",
)


Ahora si notamos diferencias.


---

### Paralelización con `concurrent.futures` y `scikit-learn`

`ProcessPoolExecutor` es la API moderna de alto nivel del módulo `concurrent.futures`. A diferencia de la clase `Process` de `multiprocessing`, permite ejecutar funciones de forma más declarativa:

```python
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(funcion, iterable)
```

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
import time

# Crear dataset para clasificación
X, y = make_classification(n_samples=10000, n_features=20, random_state=42)


# Función para entrenar un modelo con diferentes hiperparámetros
def train_model(max_depth):
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(X, y)
    return max_depth, clf.score(X, y)


depths = list(range(2, 22))

# Secuencial
start = time.time()
results_seq = [train_model(d) for d in depths]
t_seq = time.time() - start

# Paralelo con ProcessPoolExecutor
start = time.time()
with ProcessPoolExecutor() as executor:
    results_par = list(executor.map(train_model, depths))
t_par = time.time() - start

print(f"Secuencial:           {t_seq:.3f}s")
print(f"ProcessPoolExecutor:  {t_par:.3f}s")
print(f"Speedup:              {t_seq / t_par:.1f}x")

In [ ]:
plot_comparacion_tiempos(
    {"Secuencial": t_seq, "ProcessPoolExecutor": t_par},
    titulo="DecisionTree: secuencial vs ProcessPoolExecutor",
)



#### `n_jobs` en scikit-learn

Muchas herramientas de `scikit-learn` ya incorporan paralelización a través del parámetro `n_jobs`:

- `RandomForestClassifier(n_jobs=-1)` — entrena árboles en paralelo
- `GridSearchCV(..., n_jobs=-1)` — evalúa combinaciones de hiperparámetros en paralelo
- `cross_val_score(..., n_jobs=-1)` — ejecuta folds de validación cruzada en paralelo

`n_jobs=-1` usa **todos los núcleos disponibles** (internamente usa `Joblib`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
import time

X, y = make_classification(n_samples=20000, n_features=20, random_state=42)

# Sin paralelización
start = time.time()
scores_1 = cross_val_score(
    RandomForestClassifier(n_estimators=50, n_jobs=1), X, y, cv=5
)
t_1 = time.time() - start

# Con paralelización (todos los cores)
start = time.time()
scores_all = cross_val_score(
    RandomForestClassifier(n_estimators=50, n_jobs=-1), X, y, cv=5
)
t_all = time.time() - start

print(f"n_jobs=1:  {t_1:.2f}s  (accuracy: {scores_1.mean():.4f})")
print(f"n_jobs=-1: {t_all:.2f}s  (accuracy: {scores_all.mean():.4f})")
print(f"Speedup:   {t_1 / t_all:.1f}x")

In [ ]:
plot_comparacion_tiempos(
    {"n_jobs=1": t_1, "n_jobs=-1": t_all},
    titulo="sklearn RandomForest cross_val_score: n_jobs=1 vs n_jobs=-1",
)


### Asincronía y Corrutinas

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/corrutinas.png' />
</center>

En general, se utiliza más en el desarrollo web/software para no bloquear la ejecución de código al solicitar datos a un servidor externo o ejecutar un proceso muy pesado.

**Contexto en la era ML/IA:** Hoy en día, los modelos de IA (como LLMs) tardan **segundos** en generar una respuesta. Si tu sistema es **síncrono** — es decir, bloquea el hilo mientras espera la respuesta del modelo — entonces mientras un usuario espera, **ningún otro usuario puede ser atendido**. Con **asincronía**, mientras un modelo responde, el sistema puede atender otras peticiones, maximizando el uso de los recursos.

**¿Por qué importa async en ciencia de datos?**

Imagina un API que recibe peticiones de clasificación:

```python
# ❌ Síncrono: cada petición bloquea el servidor
for peticion in peticiones:
    resultado = modelo.predict(peticion)  # 2 segundos
    enviar_respuesta(resultado)           # bloquea todo
# 100 peticiones × 2s = 200 segundos de espera total

# ✅ Asíncrono: mientras uno espera, otro se procesa
async for peticion in peticiones:
    resultado = await modelo.predict(peticion)
    await enviar_respuesta(resultado)
# 100 peticiones se procesan concurrentemente
```

**Casos de uso comunes en ciencia de datos:**

- Consultas a **APIs externas** (ej: obtener datos de Twitter, OpenWeather, etc.) en paralelo sin bloquear el programa.
- **Scraping web** múltiples páginas concurrentemente.
- Descarga masiva de **archivos desde la nube** o bases de datos.
- **Servir predicciones** de modelos ML de forma concurrente.
- **Entrenamiento con hiperparámetros** donde la evaluación depende de datos externos.

> **Pregunta ❓**: ¿Qué aplicación de data science podría beneficiarse de la asincronía?

#### Ejemplo práctico: requests HTTP síncronas vs asíncronas

Comparamos **8 peticiones HTTP reales** a una API pública, primero de forma secuencial
y luego concurrente con `asyncio.gather`. Con I/O bound, el event loop puede enviar
todas las peticiones "a la vez" y esperar sus respuestas simultáneamente sin bloquear:

In [ ]:
!uv add httpx --quiet
import httpx, time

URLS = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 9)]

start = time.time()
with httpx.Client(timeout=10.0) as client:
    responses_sync = [client.get(url) for url in URLS]
t_sync = time.time() - start

print(f"Sync (secuencial):  {t_sync:.2f}s — {len(URLS)} requests")
print(f"Status: {[r.status_code for r in responses_sync]}")


In [ ]:
import asyncio, httpx, time


async def fetch_all(urls):
    async with httpx.AsyncClient(timeout=10.0) as client:
        return await asyncio.gather(*[client.get(url) for url in urls])


start = time.time()
responses_async = await fetch_all(URLS)
t_async = time.time() - start

print(f"Async (concurrente): {t_async:.2f}s — {len(responses_async)} requests")
print(f"Status: {[r.status_code for r in responses_async]}")


In [ ]:
plot_comparacion_tiempos(
    {"Sync (secuencial)": t_sync, "Async (asyncio.gather)": t_async},
    titulo=f"HTTP requests: sync vs async ({len(URLS)} URLs)",
)



> **Nota:** `asyncio` no usa múltiples núcleos — ejecuta todo en **un solo hilo**. La ganancia viene de que mientras una tarea espera I/O (respuesta de red, lectura de disco), el event loop ejecuta otra tarea. Esto es ideal para: scraping, descarga masiva de archivos, consultas a bases de datos, y cualquier operación donde el cuello de botella es la espera, no el cómputo.

---

El paralelismo local funciona bien cuando los datos caben en la memoria RAM. Pero, ¿qué pasa cuando nuestro dataset tiene **gigabytes o terabytes** de datos?

| Escenario | Herramienta recomendada |
|-----------|------------------------|
| Datos en RAM, código lento | NumPy / Numba JIT |
| Datos en RAM, tarea paralelizable | multiprocessing / Joblib |
| Datos en RAM, API más eficiente | **Polars** (lazy, multithreaded) |
| Datos > RAM, una máquina | **Dask** (particionado en disco) |
| Datos > RAM, múltiples máquinas | Dask distribuido / Spark |

Necesitamos herramientas de **procesamiento a escala**.

> **Pregunta ❓:** ¿Cuándo una máquina ya no alcanza?

**Señales de que necesitas distribuido:**

- Tu dataset no cabe en RAM
- Un solo entrenamiento tarda >1 hora
- Necesitas procesar datos en tiempo real
- Tienes múltiples fuentes de datos


---

## Procesamiento de Datos a Escala

Cuando los datos superan la capacidad de procesamiento de una sola máquina o su tamaño se acerca al límite de la RAM, existen librerías diseñadas para manejar esta situación.

<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/distributed.png'/>

El procesamiento distribuido hace referencia a la ejecución de tareas utilizando múltiples máquinas. Por lo general se refiere al trabajo con clusters de procesamiento y suele llevarse a cabo por medio de herramientas como [`Spark`](https://spark.apache.org/) o [`Dask`](https://www.dask.org/).

Diferencias entre spark y dask: https://docs.dask.org/en/stable/spark.html

En esta última sección estudiaremos `Dask` para procesar `DataFrames`.


### `Polars`

<div align="center">
<img src="https://raw.githubusercontent.com/pola-rs/polars-static/master/logos/polars_github_logo_rect_dark_name.svg" width=450>
</div>

<div align="center">
Blazingly Fast DataFrame Library 
</div>

Nueva librería alternativa a pandas enfocada en alto rendimiento y programado íntegramente en [Rust](https://www.rust-lang.org/es).

Sus principios son:

- **Rápido**: Polars está escrito desde cero, diseñado cerca de la máquina y sin dependencias externas.
- **E/S**: Soporte para todas las capas comunes de almacenamiento de datos: local, almacenamiento en la nube y bases de datos.
- **Fácil de usar**: Escriba sus consultas de la forma en que fueron concebidas. Polars, internamente, determinará la forma más eficiente de ejecutar utilizando su optimizador de consultas.
- **_Out of core_**: Polars soporta la transformación de datos fuera del núcleo con su API de streaming. Permitiéndole procesar sus resultados sin requerir que todos sus datos estén en memoria al mismo tiempo.
- **Paralelo**: Polars utiliza plenamente la potencia de su máquina dividiendo la carga de trabajo entre los núcleos de CPU disponibles sin ninguna configuración adicional.
- **Motor de consulta vectorizado**: Polars utiliza Apache Arrow, un formato de datos en columnas, para procesar sus consultas de forma vectorizada.

In [ ]:
!uv add polars

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

# Crear un dataset sintético para comparar Pandas vs Polars
np.random.seed(42)
n_rows = 5_000_000

# Columnas numéricas + categóricas
data = {
    "id": list(range(n_rows)),
    "valor_a": np.random.randn(n_rows).tolist(),
    "valor_b": np.random.randn(n_rows).tolist(),
    "valor_c": np.random.uniform(0, 100, n_rows).tolist(),
    "grupo": [f"grupo_{i % 20}" for i in range(n_rows)],
    "categoria": [f"cat_{i % 50}" for i in range(n_rows)],
}

# Crear DataFrame de pandas y Polars
df_pd = pd.DataFrame(data)
dl_pl = pl.DataFrame(data)

print(f"Pandas: {df_pd.shape}")
print(f"Polars: {dl_pl.shape}")
dl_pl.head()

In [ ]:
# Pipeline completo en Pandas
def _run_pandas():
    return (
        df_pd.query("valor_a > 0")
        .groupby("grupo")
        .agg({"valor_a": "mean", "valor_b": "sum", "id": "count"})
        .rename(columns={"id": "count"})
        .sort_values("count", ascending=False)
    )


t_pandas = %timeit -o -q _run_pandas()


In [ ]:
# Pipeline completo en Polars (eager)
def _run_polars_eager():
    return (
        dl_pl.filter(pl.col("valor_a") > 0)
        .group_by("grupo")
        .agg(
            [
                pl.col("valor_a").mean().alias("valor_a_mean"),
                pl.col("valor_b").sum().alias("valor_b_sum"),
                pl.col("id").count().alias("count"),
            ]
        )
        .sort("count", descending=True)
    )


t_polars_eager = %timeit -o -q _run_polars_eager()



#### Polars Lazy vs Eager

Polars ofrece una API **lazy** que construye un plan de ejecución optimizado antes de ejecutar. Esto puede ser significativamente más rápido porque el optimizador puede reorganizar, fusionar o eliminar operaciones:

In [ ]:
# Polars Lazy: construye plan de ejecución optimizado
def _run_polars_lazy():
    return (
        dl_pl.lazy()
        .filter(pl.col("valor_a") > 0)
        .group_by("grupo")
        .agg(
            [
                pl.col("valor_a").mean().alias("valor_a_mean"),
                pl.col("valor_b").sum().alias("valor_b_sum"),
                pl.col("id").count().alias("count"),
            ]
        )
        .sort("count", descending=True)
        .collect()
    )


t_polars_lazy = %timeit -o -q _run_polars_lazy()


In [ ]:
plot_comparacion_tiempos(
    {
        "Pandas": t_pandas,
        "Polars (eager)": t_polars_eager,
        "Polars (lazy)": t_polars_lazy,
    },
    titulo="Pipeline groupby: Pandas vs Polars eager vs Polars lazy",
)


### `Dask`
<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/dask.jpg' width=300>
</div>

Dask permite escalar procesos de Python (ya sea en un computador personal o un cluster) de manera sencilla. Provee de funcionalidades para tratar, por medio de procesamiento multi-core, con datasets masivos **que por lo general no caben en memoria.**

`Dask` fue implementado como un reemplazo de `Numpy` y `Pandas`, por ende su interfaz de usuario (API) es muy similar.
Los `DataFrames` de Dask son en términos prácticos conjuntos de `DataFrames` de pandas. Dicha separación permite ejecutar operaciones distribuidas y paralelas de forma muy eficiente.

**Principios de Dask:**

- **Lazy evaluation**: Dask construye un grafo de tareas (DAG) y solo ejecuta cuando se llama a `.compute()`. Esto permite al optimizador reorganizar, fusionar o eliminar operaciones innecesarias.
- **Particionado**: Los datos se dividen en particiones, cada una manejada por un proceso independiente. Esto permite procesar datasets que no caben en RAM.
- **API familiar**: La API de `dask.dataframe` replica la de pandas, por lo que la curva de aprendizaje es mínima.
- **Escalable**: Desde un solo computador (multi-core) hasta clusters distribuidos con cientos de nodos.
- **Integrado**: Funciona con el ecosistema PyData (NumPy, pandas, scikit-learn).


<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/dask.png' width=600 />
</div>

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2023-01/23_compilacion_y_paralelismo/dask_mimic.png' width=700>
</center>

Pueden encontrar mayor información en la página oficial del proyecto:

https://docs.dask.org/en/latest/

In [ ]:
!python -m pip install "dask[dataframe]"

#### Pandas vs Dask

##### Datos aleatorios con `pd.DataFrame`

In [ ]:
import numpy as np
import pandas as pd

# generamos datos aleatorios (disminuir en caso de no contar con suficiente memoria)
df = pd.DataFrame(np.random.random((2000000, 200)))

# generamos categorías a partir de bins para luego agregar
df[0] = pd.cut(df[0], 20)
df[1] = pd.cut(df[1], 20)
df[2] = pd.cut(df[2], 20)

df.info()

La prueba será cuanto se demora en ejecutar un `groupby(..).mean()` sobre las categorías generadas:

In [ ]:
df.groupby([0, 1, 2]).mean()

##### Inicializar `dask.dataframe`

Ahora, generamos un `Dask DataFrame` desde pandas

In [ ]:
import dask.dataframe as dd

# Dask falla al hacer groupby.mean() si los nombres de columna son enteros
# (internamente intenta c + "-count" donde c es int -> TypeError)
df.columns = df.columns.astype(str)

ddf = dd.from_pandas(df, npartitions=5)
ddf.info()

Vemamos que pasa si hacemos la misma operación que antes:

In [ ]:
ddf.groupby(["0", "1", "2"]).mean()

No produjo ningún resultados. Esto es porque Dask es **Lazy**, es decir, se ejecuta solo cuando alguien demanda su ejecución.
Esto se puede lograr a partir del método `compute()`:


In [ ]:
ddf.groupby(["0", "1", "2"]).mean().compute()

Incluso se puede visualizar como se computa la operación distribuida a través de el siguiente método:


In [ ]:
ddf.groupby(["0", "1", "2"]).mean().visualize()

##### Comparación de tiempos

En las siguientes celdas ejecutamos al comparación de tiempos

In [ ]:
t_pandas_grp = %timeit -o df.groupby(["0", "1", "2"]).mean()

In [ ]:
t_dask_grp = %timeit -o ddf.groupby(["0", "1", "2"]).mean().compute()

In [ ]:
plot_comparacion_tiempos(
    {"Pandas": t_pandas_grp, "Dask": t_dask_grp},
    titulo="groupby.mean(): Pandas vs Dask (datos en memoria)",
)


Podemos ver que **Dask** no es más rápido que **pandas** para la cantidad de datos anterior `(2000000 x 200) ~ 3.05 GB`.


> **Dask brilla cuando los datos superan la RAM disponible.** Con datasets que caben en memoria, el overhead de particionar y coordinar procesos puede superar las ganancias. Para datos que caben en RAM pero necesitan rapidez, considera **Polars** (ver más arriba).

Nuevamente, esto se debe al overhead / gasto adicional que implica lanzar varios procesos para ejecutar tareas en paralelo.

### Debo usar estos frameworks?

Si tu dataset cabe en memoria cómodamente y no es muy grande, entonces no es necesario usar `Dask`. Simplemente agregará una capa de complejidad al desarrollo.

Por otra parte, si el dataset con el cual están trabajando es masivo, entonces dichas librerías son un buen factor a considerar.

#### Comparativa: Pandas vs Polars vs Dask

| Característica | **Pandas** | **Polars** | **Dask** |
|---|---|---|---|
| Lenguaje | Python | Rust | Python |
| Memoria | En RAM | En RAM | Out-of-core / cluster |
| Paralelismo | No | Automático (multi-core) | Sí (multi-core / cluster) |
| API | Maduro | Similar a Pandas | Similar a Pandas |
| Lazy evaluation | No | Sí (optional) | Sí |
| Ecosistema | El más grande | Creciente | Integrado con PyData |
| **Cuándo usar** | Datos pequeños/medianos, ecosistema amplio | Datos grandes en RAM, máxima velocidad | Datos que no caben en RAM, clusters |

---

### Alternativas modernas al ecosistema Pandas

Además de Polars y Dask, existen otras alternativas que vale la pena conocer:

| Herramienta | Descripción | Cuándo usarla |
|---|---|---|
| **Modin** | Drop-in replacement de pandas que paraleliza automáticamente | Cuando quieres acelerar pandas sin cambiar tu código |
| **Vaex** | DataFrames out-of-core con lazy evaluation | Datasets enormes que no caben en RAM, exploración interactiva |
| **DuckDB** | Base de datos analítica embebida, SQL sobre archivos | Consultas SQL rápidas sobre CSVs/Parquets sin importar a pandas |
| **PyArrow** | Columnar in-memory format, base de Polars/Dask | Cuando necesitas máximo rendimiento con interop entre herramientas |

> **Recomendación:** Para la mayoría de los casos, la combinación **pandas + Polars** cubre el 90% de las necesidades. Usa Dask o Modin cuando los datos superen la RAM disponible.


---

## Guía de decisiones

Hemos visto múltiples herramientas y técnicas. A continuación, una guía para decidir cuándo usar cada una.


---

### ¿Cuándo NO paralelizar?

La paralelización no siempre es la respuesta. Evita paralelizar cuando:

1. **El overhead supera la ganancia:** Tareas muy ligeras o pocos datos — el costo de crear/manejar procesos supera el tiempo ahorrado.
2. **Debugging se complica:** Errores en código paralelo son más difíciles de reproducir y rastrear.
3. **Race conditions:** Si el código accede a estado compartido sin sincronización adecuada, los resultados pueden ser incorrectos.
4. **GIL (pre-3.13):** Threads no ayudan para CPU-bound en Python estándar — usa procesos en su lugar.
5. **El problema cabe en 1 núcleo:** Si tu rutina corre en tiempo razonable sin paralelismo, la complejidad adicional no justifica la optimización.

> **Regla de oro:** Primero mide, luego optimiza. No asumas que paralelizar será más rápido.


---

### Resumen de herramientas

| Herramienta | Cuándo usarla | Overhead | Escala a |
|---|---|---|---|
| **Python puro** | Prototipado, scripts | Ninguno | 1 núcleo |
| **NumPy vectorizado** | Numérico, cabe en RAM | Mínimo | 1 núcleo |
| **Numba JIT** | Loops numéricos puros | Compilación 1ª vez | 1 núcleo |
| **Joblib / ProcessPoolExecutor** | Tareas independientes | Lanzar procesos | Núcleos locales |
| **scikit-learn `n_jobs`** | Entrenamiento / grid search | Transparente | Núcleos locales |
| **Polars** | DataFrame grande, cabe en RAM | Mínimo | Núcleos locales (auto) |
| **Dask** | Dataset > RAM, cluster | Particionado | Cluster |
| **asyncio** | I/O-bound (APIs, red) | Mínimo | Concurrente |


---

> **Pregunta ❓:** ¿Cuántas operaciones por segundo puede hacer una GPU vs un CPU moderno?


**Respuesta:**

- CPU moderno (AMD Ryzen 5): ~1 billón de operaciones/segundo (1 TFLOP)
- GPU moderna (RTX 4090): ~82 billones de operaciones/segundo (82 TFLOP)
- **~80x más capacidad de cómputo** para operaciones paralelas

Pero una GPU no es siempre la respuesta: solo es más rápida cuando el problema se puede dividir en **muchas operaciones pequeñas e independientes**.


> **Pregunta ❓:** ¿Cuándo una GPU te hace más rápido y cuándo no?


**Regla práctica:**

| Escenario | GPU más rápida | CPU más rápido |
|---|---|---|
| Operaciones matriciales grandes | ✅ | |
| Entrenamiento de redes neuronales | ✅ | |
| Procesamiento de imágenes/video | ✅ | |
| Operaciones con datos pequeños (<1MB) | | ✅ |
| Lógica condicional compleja | | ✅ |
| Acceso aleatorio a memoria | | ✅ |


## GPU

### Aceleración por GPU en Ciencia de Datos

Las GPUs (*Graphics Processing Units*) fueron diseñadas originalmente para renderizar gráficos, pero su arquitectura de **miles de núcleos pequeños** las hace ideales para operaciones numéricas masivamente paralelas.

**¿Cómo funciona?**

Un CPU tiene pocos núcleos potentes (4-16), cada uno capaz de ejecutar tareas complejas. Una GPU tiene **miles de núcleos simples**, cada uno ejecutando la misma operación sobre datos diferentes simultáneamente.

```
CPU:  [Núcleo 1: tarea compleja A] [Núcleo 2: tarea compleja B]
GPU:  [Núcleo 1] [Núcleo 2] [Núcleo 3] ... [Núcleo 4096]  ← misma op sobre 4096 elementos
```

### Ventajas y desventajas de las GPUs

| Aspecto | GPU | CPU |
|---|---|---|
| **Núcleos** | Miles (1000-10000+) | Pocos (4-16) |
| **Por núcleo** | Simple, especializado | Complejo, generalista |
| **Memoria** | VRAM dedicada (8-24 GB) | Compartida con el sistema |
| **Ideal para** | Matrices, tensores, paralelismo masivo | Lógica compleja, I/O |
| **Desventaja** | Overhead de transferencia CPU↔GPU | Lento para operaciones paralelas |

### APIs y tecnologías de GPU

| API | Desarrollador | Uso |
|---|---|---|
| **CUDA** | NVIDIA | Framework más popular para GPU computing |
| **ROCm** | AMD | Alternativa open-source de NVIDIA CUDA |
| **OpenCL** | Khronos Group | Estándar abierto, multi-plataforma |
| **Metal** | Apple | GPU computing en macOS/iOS |

### Frameworks de GPU para Ciencia de Datos

| Framework | Descripción |
|---|---|
| **[JAX](https://jax.readthedocs.io/)** | NumPy diferenciable con JIT y aceleración GPU/TPU — el más versátil |
| **[CuPy](https://docs.cupy.dev/en/stable/)** | Drop-in replacement de NumPy/SciPy sobre CUDA |
| **[RAPIDS](https://rapids.ai/)** | cuDF, cuML, cuGraph — equivalentes GPU de pandas/scikit-learn |
| **[PyTorch](https://pytorch.org/)** | Framework de deep learning con aceleración GPU nativa |
| **[TensorRT](https://developer.nvidia.com/tensorrt)** | Optimización de modelos para inferencia GPU |

> **Consejo:** **JAX** es hoy la opción más recomendada para ciencia de datos de alto rendimiento: combina una API compatible con NumPy, compilación JIT (`@jax.jit`), diferenciación automática y soporte transparente para GPU/TPU — sin cambiar la lógica del código.

### Ejemplo: NumPy vs JAX JIT en multiplicación de matrices

Usamos la **multiplicación de matrices** como benchmark: es el núcleo de las redes neuronales
y uno de los casos donde la GPU muestra mayor ventaja frente al CPU.

In [ ]:
!uv add jax --quiet
# Para soporte GPU NVIDIA: pip install -U "jax[cuda12]"
# Otros soportes: https://docs.jax.dev/en/latest/installation.html

In [ ]:
import numpy as np
import time

N = 6_000  # matriz N×N float32 ≈ 144 MB
rng = np.random.default_rng(42)
A_np = rng.standard_normal((N, N)).astype(np.float32)
B_np = rng.standard_normal((N, N)).astype(np.float32)

_ = np.dot(A_np[:10, :10], B_np[:10, :10])  # warm-up BLAS

start = time.perf_counter()
for _ in range(3):
    C = np.dot(A_np, B_np)
t_numpy = (time.perf_counter() - start) / 3

print(f"NumPy (CPU): {t_numpy:.3f}s por multiplicación {N}×{N}")


In [ ]:
import jax
import jax.numpy as jnp
import time

print("Dispositivos disponibles:")
for d in jax.devices():
    print(f"  {d.device_kind}  [{d.platform.upper()}]")

A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)


@jax.jit
def matmul_jax(A, B):
    return jnp.dot(A, B)


# Warm-up: compilación JIT + primera transferencia a GPU
_ = matmul_jax(A_jax, B_jax).block_until_ready()
print("\nJIT compilado ✓")

start = time.perf_counter()
for _ in range(3):
    C_jax = matmul_jax(
        A_jax, B_jax
    ).block_until_ready()  # block_until_ready garantiza tiempo real de GPU
t_jax = (time.perf_counter() - start) / 3

dispositivo = jax.devices()[0]
plataforma = "GPU" if dispositivo.platform == "gpu" else "CPU"
label_jax = f"JAX JIT ({plataforma} · {dispositivo.device_kind})"
print(f"{label_jax}: {t_jax:.3f}s por multiplicación {N}×{N}")


In [ ]:
plot_comparacion_tiempos(
    {"NumPy (CPU)": t_numpy, label_jax: t_jax},
    titulo=f"Multiplicación de matrices {N}×{N} float32: NumPy vs JAX JIT",
)


> **`block_until_ready()`:** Las operaciones JAX en GPU son **asíncronas** — se despachan
> inmediatamente y el CPU continúa ejecutando. Sin `block_until_ready()`, `time.perf_counter()`
> mediría solo el tiempo de *despacho*, no el de *ejecución real* en GPU.